In [1]:
!pip install datasets evaluate transformers[sentencepiece]
!pip install --upgrade pip
!pip install torch
!pip install transformers[torch]
!pip install scikit-learn
!pip install matplotlib

In [2]:
from datasets import load_dataset
import csv

# 1. Define the column names according to the LIAR dataset description
col_names = [
    "id", "label_text", "statement", "subject", "speaker", 
    "job_title", "state_info", "party_affiliation", 
    "barely_true_counts", "false_counts", "half_true_counts", 
    "mostly_true_counts", "pants_on_fire_counts", "context"
]

# 2. Load local files
raw_datasets = load_dataset(
    "csv", 
    data_files={
        "train": "/home/onyxia/work/Stat_App/Data/train.tsv", 
        "validation": "/home/onyxia/work/Stat_App/Data/valid.tsv", 
        "test": "/home/onyxia/work/Stat_App/Data/test.tsv"
    }, 
    delimiter="\t", 
    column_names=col_names,
    quoting=csv.QUOTE_NONE
)

# 3. Create a mapping for the labels (Text -> Integer)
label_mapping = {
    'pants-fire': 0, 
    'false': 1, 
    'barely-true': 2, 
    'half-true': 3, 
    'mostly-true': 4, 
    'true': 5
}

def map_labels(example):
    return {'label': label_mapping[example['label_text']]}

raw_datasets = raw_datasets.map(map_labels)

print(raw_datasets)

/opt/python/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 10269
    })
    validation: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1284
    })
    test: Dataset({
        features: ['id', 'label_text', 'statement', 'subject', 'speaker', 'job_title', 'state_info', 'party_affiliation', 'barely_true_counts', 'false_counts', 'half_true_counts', 'mostly_true_counts', 'pants_on_fire_counts', 'context', 'label'],
        num_rows: 1283
    })
})


In [4]:
from transformers import BertTokenizer, BertModel
import torch
import numpy as np
from tqdm import tqdm

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def extract_cls(statements, batch_size=32):
    cls_vectors = []
    
    for i in tqdm(range(0, len(statements), batch_size)):
        batch = statements[i:i+batch_size]
        
        inputs = tokenizer(
            batch, 
            return_tensors="pt", 
            truncation=True, 
            max_length=128, 
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            cls = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            cls_vectors.append(cls)
    
    return np.concatenate(cls_vectors, axis=0)

# Extraire les CLS
print("Extraction des CLS pour train...")
x_train = extract_cls(raw_datasets["train"]["statement"])

print("Extraction des CLS pour validation...")
x_dev = extract_cls(raw_datasets["validation"]["statement"])

print("Extraction des CLS pour test...")
x_test = extract_cls(raw_datasets["test"]["statement"])

print(f"x_train shape: {x_train.shape}")
print(f"x_dev shape: {x_dev.shape}")
print(f"x_test shape: {x_test.shape}")

# Sauvegarder pour ne pas recalculer
np.save("x_train_cls.npy", x_train)
np.save("x_dev_cls.npy", x_dev)
np.save("x_test_cls.npy", x_test)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 578.47it/s, Materializing param=pooler.dense.weight]                               
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Extraction des CLS pour train...


100%|██████████| 321/321 [04:08<00:00,  1.29it/s]


Extraction des CLS pour validation...


100%|██████████| 41/41 [00:28<00:00,  1.45it/s]


Extraction des CLS pour test...


100%|██████████| 41/41 [00:28<00:00,  1.46it/s]

x_train shape: (10269, 768)
x_dev shape: (1284, 768)
x_test shape: (1283, 768)


In [5]:
import numpy as np
from sklearn.svm import LinearSVC
from collections import Counter
import time

# ============================================
# 1. CHARGER LES CLS (si déjà calculés)
# ============================================

# x_train = np.load("x_train_cls.npy")
# x_dev = np.load("x_dev_cls.npy")
# x_test = np.load("x_test_cls.npy")

# ============================================
# 2. CRÉER LES LABELS DE PARTI
# ============================================

def get_party_idx(p):
    if p == "republican":
        return 0
    elif p == "democrat":
        return 1
    else:
        return 2  # none, organization, etc.

y_train_party = np.array([get_party_idx(p) for p in raw_datasets["train"]["party_affiliation"]])
y_dev_party = np.array([get_party_idx(p) for p in raw_datasets["validation"]["party_affiliation"]])
y_test_party = np.array([get_party_idx(p) for p in raw_datasets["test"]["party_affiliation"]])

print("Distribution des partis:")
print("Train:", Counter(y_train_party))
print("Dev:", Counter(y_dev_party))
print("Test:", Counter(y_test_party))

# Labels de la tâche principale (véracité)
y_train_task = np.array(raw_datasets["train"]["label"])
y_dev_task = np.array(raw_datasets["validation"]["label"])
y_test_task = np.array(raw_datasets["test"]["label"])

# ============================================
# 3. FONCTIONS INLP
# ============================================

def get_nullspace_projection(W):
    """Calcule la projection sur le null-space de W avec SVD"""
    U, S, Vt = np.linalg.svd(W, full_matrices=False)
    rank = np.sum(S > 1e-10)
    rowspace_basis = Vt[:rank]
    P = np.eye(W.shape[1]) - rowspace_basis.T @ rowspace_basis
    return P

def get_debiasing_projection(X_train, Y_train_party, X_dev, Y_dev_party, num_clfs=300, min_acc=0.35):
    """
    Applique INLP pour supprimer l'information du parti politique
    
    X_train, X_dev : les [CLS] de BERT (N x 768)
    Y_train_party, Y_dev_party : les labels de parti (0=republican, 1=democrat, 2=none)
    num_clfs : nombre maximum d'itérations
    min_acc : accuracy minimale pour continuer (hasard = 0.33 pour 3 classes)
    """
    
    dim = X_train.shape[1]
    P = np.eye(dim)
    X_train_proj = X_train.copy()
    X_dev_proj = X_dev.copy()
    
    rowspace_projections = []
    Ws = []
    
    start = time.time()
    
    for i in range(num_clfs):
        # Entraîner un SVM linéaire pour prédire le parti
        clf = LinearSVC(penalty='l2', C=0.01, fit_intercept=True, class_weight=None, dual=False, max_iter=10000)
        clf.fit(X_train_proj, Y_train_party)
        
        # Évaluer l'accuracy
        acc_train = clf.score(X_train_proj, Y_train_party)
        acc_dev = clf.score(X_dev_proj, Y_dev_party)
        print(f"Iteration {i+1}/{num_clfs}, Train Acc: {acc_train:.4f}, Dev Acc: {acc_dev:.4f}")
        
        # Si l'accuracy est proche du hasard (0.33 pour 3 classes), on arrête
        if acc_dev < min_acc:
            print(f"Accuracy proche du hasard ({acc_dev:.4f}), arrêt.")
            break
        
        # Récupérer les poids du classificateur
        W = clf.coef_
        Ws.append(W)
        
        # Calculer la projection sur le null-space
        P_i = get_nullspace_projection(W)
        rowspace_projections.append(np.eye(dim) - P_i)
        
        # Mettre à jour la projection finale
        P = P_i @ P
        
        # Projeter les données
        X_train_proj = X_train_proj @ P_i.T
        X_dev_proj = X_dev_proj @ P_i.T
    
    print(f"Temps total: {time.time() - start:.2f}s")
    print(f"Nombre d'itérations: {len(Ws)}")
    
    return P, rowspace_projections, Ws

# ============================================
# 4. APPLIQUER INLP
# ============================================

print("\nApplication d'INLP pour supprimer le biais de parti politique...")
P, rowspace_projections, Ws = get_debiasing_projection(
    x_train, y_train_party, 
    x_dev, y_dev_party, 
    num_clfs=10
)

# ============================================
# 5. PROJETER LES DONNÉES
# ============================================

x_train_debiased = x_train @ P.T
x_dev_debiased = x_dev @ P.T
x_test_debiased = x_test @ P.T

# Sauvegarder
np.save("x_train_debiased.npy", x_train_debiased)
np.save("x_dev_debiased.npy", x_dev_debiased)
np.save("x_test_debiased.npy", x_test_debiased)
np.save("P_party.npy", P)

print(f"\nProjection matrix P shape: {P.shape}")
print("Données debiasées sauvegardées.")

# ============================================
# 6. VÉRIFIER QUE LE PARTI N'EST PLUS PRÉDICTIBLE
# ============================================

print("\nVérification après debiasing:")
clf_verif = LinearSVC(penalty='l2', C=0.01, fit_intercept=True, dual=False, max_iter=10000)
clf_verif.fit(x_train_debiased, y_train_party)
acc_after = clf_verif.score(x_dev_debiased, y_dev_party)
print(f"Accuracy prédiction parti après debiasing: {acc_after:.4f} (hasard = 0.33)")

Distribution des partis:
Train: Counter({np.int64(0): 4510, np.int64(1): 3345, np.int64(2): 2414})
Dev: Counter({np.int64(0): 597, np.int64(1): 395, np.int64(2): 292})
Test: Counter({np.int64(0): 580, np.int64(1): 410, np.int64(2): 293})

Application d'INLP pour supprimer le biais de parti politique...
Iteration 1/10, Train Acc: 0.5647, Dev Acc: 0.4907
Iteration 2/10, Train Acc: 0.5101, Dev Acc: 0.4914
Iteration 3/10, Train Acc: 0.4865, Dev Acc: 0.4899
Iteration 4/10, Train Acc: 0.4729, Dev Acc: 0.4665
Iteration 5/10, Train Acc: 0.4670, Dev Acc: 0.4572
Iteration 6/10, Train Acc: 0.4588, Dev Acc: 0.4494
Iteration 7/10, Train Acc: 0.4564, Dev Acc: 0.4540
Iteration 8/10, Train Acc: 0.4518, Dev Acc: 0.4447
Iteration 9/10, Train Acc: 0.4464, Dev Acc: 0.4439
Iteration 10/10, Train Acc: 0.4437, Dev Acc: 0.4525
Temps total: 136.57s
Nombre d'itérations: 10

Projection matrix P shape: (768, 768)
Données debiasées sauvegardées.

Vérification après debiasing:
Accuracy prédiction parti après debias